In [1]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    precision_recall_curve,
    average_precision_score,
    roc_curve,
    roc_auc_score
)

---

# Load all metrics for all models in a dictionary

In [2]:
load_model_bool = False
load_metrics_bool = True


parent_folder = "classification_ring/models"

models_dict = {
    "logistic": parent_folder + "/logistic_classifier",
    "random_forest": parent_folder + "/random_forest",
    "xgboost": parent_folder + "/xgboost",
}


if load_model_bool:
    models = {}
    import pickle
    for model_name, model_folder in models_dict.items():
        model_path = f"{model_folder}/{model_name}_model.pkl"
        with open(model_path, "rb") as f:
            models[model_name] = pickle.load(f)
        print(f"Model loaded from {model_path}")


if load_metrics_bool:
    metrics_dict = {}
    for model_name, model_folder in models_dict.items():
        # Postprocessed results paths
        postprocessed_threshold_results_path = f"{model_folder}/postprocessed_threshold_results.parquet"
        postprocessed_curve_results_path = f"{model_folder}/postprocessed_curve_results.parquet"
        # Original results paths
        threshold_results_path = f"{model_folder}/threshold_results.parquet"
        curve_results_path = f"{model_folder}/curve_results.parquet"

        metrics_dict[model_name] = {
            "postprocessed_threshold_results": pd.read_parquet(postprocessed_threshold_results_path),
            "postprocessed_curve_results": pd.read_parquet(postprocessed_curve_results_path),
            "threshold_results": pd.read_parquet(threshold_results_path),
            "curve_results": pd.read_parquet(curve_results_path)
        }
        print(f"Metrics loaded for {model_name} from {model_folder}")

Metrics loaded for logistic from classification_ring/models/logistic_classifier
Metrics loaded for random_forest from classification_ring/models/random_forest
Metrics loaded for xgboost from classification_ring/models/xgboost


# Compare same model, with and without post-processing

In [3]:
def summarize_threshold_df(threshold_df, fixed_threshold=0.5):
    """
    Summarizes the threshold results for each class in the given DataFrame.
    Identifies the best threshold as the one maximizing the MCC and also
    retrieves the MCC and balanced accuracy at a fixed threshold (default is 0.5).
    """
    rows = []

    for class_name, class_df in threshold_df.groupby("class"):

        fixed_row = class_df[
            np.isclose(class_df["threshold"], fixed_threshold)
        ].iloc[0]

        best_row = (
            class_df
            .sort_values("MCC", ascending=False)
            .iloc[0]
        )

        rows.append({
            "class": class_name,
            "MCC_at_0_5": fixed_row["MCC"],
            "BA_at_0_5": fixed_row["balanced_accuracy"],
            "best_MCC": best_row["MCC"],
            "best_threshold": best_row["threshold"],
            "BA_at_best_MCC": best_row["balanced_accuracy"]
        })

    return pd.DataFrame(rows)






def compare_baseline_vs_postprocessing(metrics_dict):
    """
    Compares the baseline and postprocessed metrics for each model in the given metrics dictionary.
    Outputs a dictionary containing comparison DataFrames for each model, including deltas for key metrics.
    """
    
    results_comparison_dict = {}

    label_cols = [
        "HBOND",
        "VDW",
        "IONIC",
        "PIPISTACK",
        "PICATION",
        "SSBOND",
        "PIHBOND"
    ]
    fixed_threshold = 0.5

    for model_name, model_metrics in metrics_dict.items():

        print(f"Processing metrics for model: {model_name}")

        baseline_threshold_results_df = model_metrics["threshold_results"]
        postprocessed_threshold_results_df = model_metrics["postprocessed_threshold_results"]
        baseline_curve_results_df = model_metrics["curve_results"]
        postprocessed_curve_results_df = model_metrics["postprocessed_curve_results"]

        # ------------------------------------------------------------
        # 2. Curve summaries: ROC-AUC, AP
        # ------------------------------------------------------------

        baseline_curve_summary_df = (
            baseline_curve_results_df[
                ["class", "average_precision", "ROC_AUC"]
            ]
        )

        postprocessed_curve_summary_df = (
            postprocessed_curve_results_df[
                ["class", "average_precision", "ROC_AUC"]
            ]
        )


        # ------------------------------------------------------------
        # 3. Threshold summaries from existing threshold grids
        #    - fixed MCC@0.5
        #    - best MCC and threshold
        # ------------------------------------------------------------


        baseline_threshold_summary_df = summarize_threshold_df(
            baseline_threshold_results_df,
            fixed_threshold=fixed_threshold
        )

        postprocessed_threshold_summary_df = summarize_threshold_df(
            postprocessed_threshold_results_df,
            fixed_threshold=fixed_threshold
        )


        # ------------------------------------------------------------
        # 4. Merge everything
        # ------------------------------------------------------------

        metric_comparison_df = (
            baseline_curve_summary_df
            .merge(
                postprocessed_curve_summary_df,
                on="class",
                suffixes=("_baseline", "_postprocessed")
            )
            .merge(
                baseline_threshold_summary_df,
                on="class"
            )
            .rename(columns={
                "MCC_at_0_5": "MCC_at_0_5_baseline",
                "BA_at_0_5": "BA_at_0_5_baseline",
                "best_MCC": "best_MCC_baseline",
                "best_threshold": "best_threshold_baseline",
                "BA_at_best_MCC": "BA_at_best_MCC_baseline"
            })
            .merge(
                postprocessed_threshold_summary_df,
                on="class"
            )
            .rename(columns={
                "MCC_at_0_5": "MCC_at_0_5_postprocessed",
                "BA_at_0_5": "BA_at_0_5_postprocessed",
                "best_MCC": "best_MCC_postprocessed",
                "best_threshold": "best_threshold_postprocessed",
                "BA_at_best_MCC": "BA_at_best_MCC_postprocessed"
            })
        )


        # ------------------------------------------------------------
        # 5. Add deltas
        # ------------------------------------------------------------

        for metric in [
            "ROC_AUC",
            "average_precision",
            "MCC_at_0_5",
            "BA_at_0_5",
            "best_MCC",
            "BA_at_best_MCC"
        ]:

            metric_comparison_df[f"delta_{metric}"] = (
                metric_comparison_df[f"{metric}_postprocessed"]
                - metric_comparison_df[f"{metric}_baseline"]
            )


        # ------------------------------------------------------------
        # 6. Preserve original label order
        # ------------------------------------------------------------

        metric_comparison_df["class"] = pd.Categorical(
            metric_comparison_df["class"],
            categories=label_cols,
            ordered=True
        )

        metric_comparison_df = (
            metric_comparison_df
            .sort_values("class")
            .reset_index(drop=True)
        )

        # ------------------------------------------------------------
        # 7. Store the final comparison dataframe for this model
        # ------------------------------------------------------------
        results_comparison_dict[model_name] = metric_comparison_df
    return results_comparison_dict

In [4]:
baseline_vs_postprocessing_dict = compare_baseline_vs_postprocessing(metrics_dict)

Processing metrics for model: logistic
Processing metrics for model: random_forest
Processing metrics for model: xgboost


In [5]:
delta_cols = [col for col in baseline_vs_postprocessing_dict["logistic"].columns if col.startswith("delta_")]

baseline_vs_postprocessing_dict["logistic"][["class"] + delta_cols]

,class,delta_ROC_AUC,delta_average_precision,delta_MCC_at_0_5,delta_BA_at_0_5,delta_best_MCC,delta_BA_at_best_MCC
0,HBOND,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
1,VDW,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
2,IONIC,0.017882,1.836556e-01,0.153580,0.028323,0.147244,0.034257
3,PIPISTACK,0.014428,3.170207e-01,0.284835,0.019944,0.229188,0.021159
4,PICATION,0.021572,2.037421e-01,0.137669,0.021548,0.131872,0.022834
5,SSBOND,0.000000,-1.110223e-16,0.008191,0.000023,0.000000,0.000000
6,PIHBOND,0.018008,7.953368e-04,0.007226,0.015717,0.005356,0.145287


In [6]:
baseline_vs_postprocessing_dict["random_forest"][["class"] + delta_cols]

,class,delta_ROC_AUC,delta_average_precision,delta_MCC_at_0_5,delta_BA_at_0_5,delta_best_MCC,delta_BA_at_best_MCC
0,HBOND,0.000000e+00,0.000000e+00,0.0,0.0,0.000000,0.000000
1,VDW,0.000000e+00,0.000000e+00,0.0,0.0,0.000000,0.000000
2,IONIC,1.497963e-05,5.779259e-05,0.0,0.0,0.000000,0.000000
3,PIPISTACK,-1.110223e-16,-1.110223e-16,0.0,0.0,0.000000,0.000000
4,PICATION,3.259924e-05,2.060812e-04,0.0,0.0,0.000163,0.000008
5,SSBOND,0.000000e+00,0.000000e+00,0.0,0.0,0.000000,0.000000
6,PIHBOND,1.052621e-03,1.330435e-04,0.0,0.0,0.000000,0.000000


Convert to latex table

In [7]:
# ============================================================
# Wide report tables
# ============================================================

rows = {
    "ROC-AUC": ("ROC_AUC", 3),
    "Average precision": ("average_precision", 3),
    "MCC@0.5": ("MCC_at_0_5", 3),
    "BA@0.5": ("BA_at_0_5", 3),
    "Best MCC": ("best_MCC", 3),
    "BA@best MCC": ("BA_at_best_MCC", 3),
    "Best threshold": ("best_threshold", 2),
}


def build_metric_table(metric_comparison_df, selected_classes, restricted=False):

    table = pd.DataFrame(index=rows.keys())

    for class_name in selected_classes:

        r = metric_comparison_df[metric_comparison_df["class"] == class_name].iloc[0] # take the row corresponding to the current class

        for label, (metric, digits) in rows.items():


            if label == "Best threshold":
                value = (
                    f"{r[f'{metric}_baseline']:.{digits}f}"
                    f" / {r[f'{metric}_postprocessed']:.{digits}f}"
                )
            else:
                value = (
                    f"{r[f'{metric}_baseline']:.{digits}f}"
                    f" / {r[f'{metric}_postprocessed']:.{digits}f}"
                    f" ({r[f'delta_{metric}']:+.{digits}f})"
                )

            table.loc[label, class_name] = value

    return table.reset_index(names="metric")

In [12]:
metric_comparison_df = baseline_vs_postprocessing_dict["xgboost"]



unrestricted_classes = ["HBOND", "VDW"]
restricted_classes = ["IONIC", "PIPISTACK", "PICATION", "SSBOND", "PIHBOND"]


unrestricted_metric_table_df = build_metric_table(
    metric_comparison_df,
    unrestricted_classes,
    restricted=False
)

restricted_metric_table_df = build_metric_table(
    metric_comparison_df,
    restricted_classes,
    restricted=True
)

latex_unrestricted_metric_table = unrestricted_metric_table_df.to_latex(
    index=False,
    escape=True,
    column_format="l" + "c" * len(unrestricted_classes)
)

latex_restricted_metric_table = restricted_metric_table_df.to_latex(
    index=False,
    escape=True,
    column_format="l" + "c" * len(restricted_classes)
)

print(latex_unrestricted_metric_table)
print()
print(latex_restricted_metric_table)

\begin{tabular}{lcc}
\toprule
metric & HBOND & VDW \\
\midrule
ROC-AUC & 0.752 / 0.752 (+0.000) & 0.645 / 0.645 (+0.000) \\
Average precision & 0.879 / 0.879 (+0.000) & 0.651 / 0.651 (+0.000) \\
MCC@0.5 & 0.328 / 0.328 (+0.000) & 0.196 / 0.196 (+0.000) \\
BA@0.5 & 0.613 / 0.613 (+0.000) & 0.598 / 0.598 (+0.000) \\
Best MCC & 0.351 / 0.351 (+0.000) & 0.197 / 0.197 (+0.000) \\
BA@best MCC & 0.653 / 0.653 (+0.000) & 0.597 / 0.597 (+0.000) \\
Best threshold & 0.61 / 0.61 & 0.52 / 0.52 \\
\bottomrule
\end{tabular}


\begin{tabular}{lccccc}
\toprule
metric & IONIC & PIPISTACK & PICATION & SSBOND & PIHBOND \\
\midrule
ROC-AUC & 0.986 / 0.987 (+0.000) & 0.997 / 0.997 (+0.000) & 0.993 / 0.993 (+0.000) & 1.000 / 1.000 (+0.000) & 0.954 / 0.956 (+0.002) \\
Average precision & 0.559 / 0.559 (+0.000) & 0.850 / 0.850 (+0.000) & 0.366 / 0.366 (+0.000) & 0.939 / 0.939 (+0.000) & 0.016 / 0.017 (+0.000) \\
MCC@0.5 & 0.440 / 0.440 (+0.000) & 0.853 / 0.853 (+0.000) & 0.185 / 0.185 (+0.000) & 0.902 / 0.902 

---
# All models, key metrics compared

In [9]:
def make_model_comparison_table(metrics_dict, which = "postprocessed"):
    """
    Takes the the metrics df for each of the model, and returns a single df with the metrics for each model, for each class.
    The 'which' parameter can be either 'postprocessed' or 'baseline', to select which version to include in the comparison table.
    """
    rows = []

    baseline_vs_postprocessing_dict = compare_baseline_vs_postprocessing(metrics_dict)
    
    if which == "postprocessed":
        for model_name, df in baseline_vs_postprocessing_dict.items():
            
            tmp = df[[
                "class",
                "average_precision_postprocessed",
                "ROC_AUC_postprocessed",
                "MCC_at_0_5_postprocessed",
                "BA_at_0_5_postprocessed",
                "best_MCC_postprocessed",
                "best_threshold_postprocessed"
            ]].copy()

            tmp.insert(0, "model", model_name)

            tmp = tmp.rename(columns={
                "average_precision_postprocessed": "AP",
                "ROC_AUC_postprocessed": "ROC-AUC",
                "MCC_at_0_5_postprocessed": "MCC@0.5",
                "BA_at_0_5_postprocessed": "BA@0.5",
                "best_MCC_postprocessed": "best MCC",
                "best_threshold_postprocessed": "best threshold"
            })

            rows.append(tmp)

    if which == "baseline":
        for model_name, df in baseline_vs_postprocessing_dict.items():
            
            tmp = df[[
                "class",
                "average_precision_baseline",
                "ROC_AUC_baseline",
                "MCC_at_0_5_baseline",
                "BA_at_0_5_baseline",
                "best_MCC_baseline",
                "best_threshold_baseline"
            ]].copy()

            tmp.insert(0, "model", model_name)

            tmp = tmp.rename(columns={
                "average_precision_baseline": "AP",
                "ROC_AUC_baseline": "ROC-AUC",
                "MCC_at_0_5_baseline": "MCC@0.5",
                "BA_at_0_5_baseline": "BA@0.5",
                "best_MCC_baseline": "best MCC",
                "best_threshold_baseline": "best threshold"
            })

            rows.append(tmp)

    if which == "both":
        for model_name, df in baseline_vs_postprocessing_dict.items():
            
            tmp = df[[
                "class",
                "average_precision_baseline",
                "ROC_AUC_baseline",
                "MCC_at_0_5_baseline",
                "BA_at_0_5_baseline",
                "best_MCC_baseline",
                "best_threshold_baseline",
                "average_precision_postprocessed",
                "ROC_AUC_postprocessed",
                "MCC_at_0_5_postprocessed",
                "BA_at_0_5_postprocessed",
                "best_MCC_postprocessed",
                "best_threshold_postprocessed"
            ]].copy()

            tmp.insert(0, "model", model_name)

            tmp = tmp.rename(columns={
                "average_precision_baseline": "AP_baseline",
                "ROC_AUC_baseline": "ROC-AUC_baseline",
                "MCC_at_0_5_baseline": "MCC@0.5_baseline",
                "BA_at_0_5_baseline": "BA@0.5_baseline",
                "best_MCC_baseline": "best MCC_baseline",
                "best_threshold_baseline": "best threshold_baseline",
                "average_precision_postprocessed": "AP_postprocessed",
                "ROC_AUC_postprocessed": "ROC-AUC_postprocessed",
                "MCC_at_0_5_postprocessed": "MCC@0.5_postprocessed",
                "BA_at_0_5_postprocessed": "BA@0.5_postprocessed",
                "best_MCC_postprocessed": "best MCC_postprocessed",
                "best_threshold_postprocessed": "best threshold_postprocessed"
            })

            rows.append(tmp)
    comparison_df = pd.concat(rows, ignore_index=True)

    label_order = [
        "HBOND",
        "VDW",
        "IONIC",
        "PIPISTACK",
        "PICATION",
        "SSBOND",
        "PIHBOND"
    ]

    comparison_df["class"] = pd.Categorical(
        comparison_df["class"],
        categories=label_order,
        ordered=True
    )

    comparison_df = (
        comparison_df
        .sort_values(["class", "model"])
        .reset_index(drop=True)
    )

    # Round numeric columns to 3 decimal places for better readability
    numeric_cols = comparison_df.select_dtypes(include=[np.number]).columns
    comparison_df[numeric_cols] = comparison_df[numeric_cols].round(3)


    return comparison_df

In [13]:
model_comparison_df = make_model_comparison_table(
    metrics_dict,
    which="postprocessed"
)

model_comparison_df

Processing metrics for model: logistic
Processing metrics for model: random_forest
Processing metrics for model: xgboost


,model,class,AP,ROC-AUC,MCC@0.5,BA@0.5,best MCC,best threshold
0,logistic,HBOND,0.828,0.668,0.219,0.621,0.228,0.40
1,random_forest,HBOND,0.889,0.774,0.366,0.637,0.384,0.63
2,xgboost,HBOND,0.879,0.752,0.328,0.613,0.351,0.61
3,logistic,VDW,0.586,0.592,0.129,0.565,0.131,0.49
4,random_forest,VDW,0.650,0.647,0.197,0.598,0.199,0.55
5,xgboost,VDW,0.651,0.645,0.196,0.598,0.197,0.52
6,logistic,IONIC,0.499,0.984,0.554,0.969,0.567,0.70
7,random_forest,IONIC,0.573,0.986,0.450,0.664,0.582,0.19
8,xgboost,IONIC,0.559,0.987,0.440,0.657,0.585,0.21
9,logistic,PIPISTACK,0.776,0.996,0.853,0.995,0.853,0.66


In [14]:
latex_table = model_comparison_df.to_latex(
    index=False,
    escape=False,
    column_format="llcccccc",
    float_format="%.3f"
)

latex_table = latex_table.replace("\\begin{tabular}", "\\begin{tabular}")
print(latex_table)

\begin{tabular}{llcccccc}
\toprule
model & class & AP & ROC-AUC & MCC@0.5 & BA@0.5 & best MCC & best threshold \\
\midrule
logistic & HBOND & 0.828 & 0.668 & 0.219 & 0.621 & 0.228 & 0.400 \\
random_forest & HBOND & 0.889 & 0.774 & 0.366 & 0.637 & 0.384 & 0.630 \\
xgboost & HBOND & 0.879 & 0.752 & 0.328 & 0.613 & 0.351 & 0.610 \\
logistic & VDW & 0.586 & 0.592 & 0.129 & 0.565 & 0.131 & 0.490 \\
random_forest & VDW & 0.650 & 0.647 & 0.197 & 0.598 & 0.199 & 0.550 \\
xgboost & VDW & 0.651 & 0.645 & 0.196 & 0.598 & 0.197 & 0.520 \\
logistic & IONIC & 0.499 & 0.984 & 0.554 & 0.969 & 0.567 & 0.700 \\
random_forest & IONIC & 0.573 & 0.986 & 0.450 & 0.664 & 0.582 & 0.190 \\
xgboost & IONIC & 0.559 & 0.987 & 0.440 & 0.657 & 0.585 & 0.210 \\
logistic & PIPISTACK & 0.776 & 0.996 & 0.853 & 0.995 & 0.853 & 0.660 \\
random_forest & PIPISTACK & 0.847 & 0.997 & 0.843 & 0.966 & 0.855 & 0.240 \\
xgboost & PIPISTACK & 0.850 & 0.997 & 0.853 & 0.984 & 0.856 & 0.360 \\
logistic & PICATION & 0.287 & 0.990 & 0